# 🤖 Fase 3: Agente de Triaje de Emails
## Cerebro del agente — TechPyme Atención al Cliente

---

En este notebook construimos el **agente completo** que lee correos de clientes, busca en la base de conocimiento RAG y toma decisiones de forma autónoma.

### ¿Qué hace el agente en cada correo?
```
Correo entrante
      │
      ▼
 [Retriever]  →  Busca en ChromaDB los 2 fragmentos más relevantes
      │
      ▼
  [LLM local]  →  Razona con el correo + contexto recuperado
      │
      ├── ¿Respuesta en el contexto? →          Redacta respuesta automática
      │                                             │
      │                                             ▼
      │                                         Guarda en correos_respondidos.csv
      └── ¿No hay info / cliente enfadado? →    ESCALAR_A_HUMANO
                                                    │
                                                    ▼
                                                Guarda en correos_escalados.csv
```

### Requisitos previos
- ✅ **Fase 2 completada**: la carpeta `../F2 - RAG/chroma_db` debe existir
- ✅ **Ollama instalado** y ejecutándose en segundo plano
- ✅ **Modelo descargado**: ejecutar `ollama pull llama3` en terminal

---
## Paso 1 — Instalación de librerías adicionales

Necesitamos `langchain-ollama` para conectar con el modelo local.

> ℹ️ Si ya instalaste las librerías en la Fase 2, solo necesitas el paquete nuevo.

In [ ]:
%pip install langchain-ollama langchain langchain-chroma langchain-huggingface sentence-transformers

---
## Paso 2 — Verificar que Ollama está activo

Antes de continuar, comprobamos que el servidor de Ollama responde correctamente.

In [ ]:
import subprocess
import sys

print("Verificando Ollama...")

try:
    result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(" Ollama está activo.")
        print(" Modelos disponibles:")
        for line in result.stdout.strip().split("\n"):
            print(f"   {line}")
    else:
        print(" Ollama no responde. Ábre una terminal y ejecuta: 'ollama serve' .")
        sys.exit(1)
except FileNotFoundError:
    print(" Ollama no está instalado. Descárgalo en: 'https://ollama.ai' .")
    sys.exit(1)
except subprocess.TimeoutExpired:
    print(" Timeout al contactar Ollama. Asegúrate de que está ejecutándose en segundo plano.")

---
## Paso 3 — Bandeja de entrada simulada (`correos.json`)

Para no depender de conexiones reales a Gmail/Outlook en clase, simulamos la bandeja de entrada con un archivo JSON.

Incluimos **4 casos de prueba** que cubren todos los escenarios del agente:

| # | Remitente | Escenario esperado |
|---|-----------|--------------------|
| 1 | carlos | Pregunta de envío → respuesta automática |
| 2 | laura | Producto roto + enfadada → escalar |
| 3 | pedro | Soporte externo (Amazon) → escalar |
| 4 | marta | Pregunta de horario → respuesta automática |

---
## Paso 4 — Conectar con la Base de Conocimiento RAG

Cargamos la ChromaDB que creamos en la **Fase 2**. El agente la usará para buscar información relevante antes de responder.

### Parámetros clave:
- **`search_kwargs={"k": 2}`**: Ajustamos a 2 el número de fragmentos que nos devolverá el RAG

In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Ruta a la base de datos creada en la Fase 2
DIRECTORIO_DB = "../F2 - RAG/chroma_db"

# Verificamos que la base de datos existe
if not os.path.exists(DIRECTORIO_DB):
    raise FileNotFoundError(
        f" No se encontró la base de datos en '{DIRECTORIO_DB}'.\n"
        "  Ejecuta primero el notebook de la Fase 2 para crearla."
    )

print("1. Cargando modelo de embeddings (debe ser el mismo que en Fase 2)...")
embeddings_model = HuggingFaceEmbeddings(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

print("2.  Conectando con la base de conocimiento RAG...")
vectorstore = Chroma(
    persist_directory=DIRECTORIO_DB,
    embedding_function=embeddings_model
)

# Configuramos el recuperador: traerá los 2 fragmentos más relevantes por consulta
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print(f" Base de conocimiento cargada. Fragmentos disponibles: {vectorstore._collection.count()}")

---
## Paso 5 — Despertar la IA local (Ollama + LLaMA3)

Conectamos con el modelo de lenguaje que corre localmente a través de Ollama.

### Parámetros clave:
- **`model="llama3"`**: El modelo a usar (debe estar descargado con `ollama pull llama3`)
- **`temperature=0`**: Sin creatividad — el modelo responde con precisión, sin inventar

In [ ]:
from langchain_ollama import ChatOllama

print(" Despertando a la IA local (Ollama)...")

# Cambia 'llama3' por el modelo que tengas descargado (ej: 'mistral', 'llama3.2', etc.)
MODELO = "llama3"

llm = ChatOllama(
    model=MODELO,
    temperature=0  # Sin creatividad: respuestas precisas y reproducibles
)

# Hacemos un ping rápido para confirmar que el modelo responde
respuesta_test = llm.invoke("Di solo 'OK' en español.")
print(f" Modelo '{MODELO}' activo. Respuesta de prueba: '{respuesta_test.content.strip()}'")

---
## Paso 6 — Definir el System Prompt (Las reglas del agente)

El **System Prompt** es el conjunto de instrucciones que define el comportamiento del agente. Es como el "manual de empleado" de la IA.

### Reglas del agente TechPyme:
1. **Solo usa el contexto RAG** para responder — nunca inventa información
2. **Escala a humano** si el cliente está enfadado o el problema es complejo
3. **Escala a humano** si la respuesta no está en el contexto
4. Si puede responder: redacta un correo **amable y profesional**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("3. Configurando las instrucciones del Agente (System Prompt)...")

instrucciones_agente = """
Eres un asistente de atención al cliente de la tienda online TechPyme.
Tu objetivo es leer el correo del cliente y responder basándote ÚNICAMENTE en el siguiente
contexto de la empresa:

<contexto>
{context}
</contexto>

REGLAS ESTRICTAS (síguellas en orden):
1. Si el cliente expresa enfado, frustración, urgencia extrema o reporta un producto roto/defectuoso,
   responde EXACTAMENTE con la palabra: ESCALAR_A_HUMANO
2. Si la respuesta a la pregunta del cliente NO está en el contexto proporcionado,
   responde EXACTAMENTE con la palabra: ESCALAR_A_HUMANO
3. Si puedes responder con la información del contexto:
   - Comienza con un saludo amable
   - Responde de forma clara, concisa y profesional
   - Cierra con una despedida cordial firmando como "El equipo de TechPyme"

IMPORTANTE: No añadas explicaciones sobre por qué escalas. Si escalas, di SOLO: ESCALAR_A_HUMANO
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", instrucciones_agente),
    ("human", "Asunto del correo: {asunto}\n\nMensaje del cliente:\n{mensaje}")
])

print(" System Prompt configurado.")

---
## Paso 7 — Construir la Cadena RAG (El flujo completo del agente)

Unimos todas las piezas en una **cadena de ejecución**:

```
correo  →  [retriever]  →  [prompt + contexto]  →  [LLM]  →  respuesta
```

- `create_stuff_documents_chain`: inyecta los documentos recuperados en el prompt
- `create_retrieval_chain`: orquesta todo el flujo RAG

In [ ]:
# Función auxiliar: convierte los documentos recuperados en texto plano
def formatear_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Cadena RAG con LCEL — compatible con langchain-core 1.x (sin langchain.chains)
rag_chain = (
    {
        "context": (lambda x: x["mensaje"]) | retriever | formatear_docs,
        "asunto":  RunnablePassthrough() | (lambda x: x["asunto"]),
        "mensaje": RunnablePassthrough() | (lambda x: x["mensaje"])
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(" Cadena RAG construida con LCEL. El agente está listo.")
print()
print("  Arquitectura del agente:")
print("  correo  →  [ChromaDB retriever]  →  [LLaMA3 local]  →  decisión")

---
## Paso 8 — Ejecutar el Agente de Triaje

El bucle de automatización lee cada correo del JSON, invoca el agente y toma la decisión:
- ** Respuesta automática** → la muestra por pantalla
- ** ESCALAR_A_HUMANO** → lo registra en `correos_escalados.csv`

>  Puede tardar unos segundos por correo dependiendo del hardware.

In [ ]:
import csv
import json
from datetime import datetime

ARCHIVO_ESCALADOS = "correos_escalados.csv"
ARCHIVO_RESPONDIDOS  = "correos_respondidos.csv"

# Creamos los CSV con cabecera si no existen
if not os.path.exists(ARCHIVO_ESCALADOS):
    with open(ARCHIVO_ESCALADOS, mode='w', newline='', encoding='utf-8') as f:
        escritor = csv.writer(f)
        escritor.writerow(["timestamp", "id", "remitente", "asunto", "cuerpo", "motivo"])
    print(f" Creado archivo '{ARCHIVO_ESCALADOS}' con cabecera.")

if not os.path.exists(ARCHIVO_RESPONDIDOS):
        with open(ARCHIVO_RESPONDIDOS, mode="w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(["timestamp", "id", "remitente", "asunto", "cuerpo", "respuesta"])
        print(f"Creado archivo '{ARCHIVO_RESPONDIDOS}' con cabecera.")

# Leemos los correos simulados
with open('correos.json', 'r', encoding='utf-8') as f:
    correos = json.load(f)

print(f" Procesando {len(correos)} correos entrantes...")
print("=" * 60)

# ──────────────────────────────────────────────
#  BUCLE DE AUTOMATIZACIÓN — El agente trabajando
# ──────────────────────────────────────────────
for correo in correos:
    print(f"\n Correo #{correo['id']} | De: {correo['remitente']}")
    print(f"   Asunto: {correo['asunto']}")
    print(f"   Procesando...", end=" ", flush=True)

    # La cadena LCEL devuelve str directamente (no un dict)
    decision_ia = rag_chain.invoke({
        "asunto":  correo["asunto"],
        "mensaje": correo["cuerpo"]
    }).strip()

    if "ESCALAR_A_HUMANO" in decision_ia:
        print("   DECISIÓN: Derivado a agente humano.")
        print("   Guardando en correos_escalados.csv...")

        with open(ARCHIVO_ESCALADOS, mode='a', newline='', encoding='utf-8') as f:
            escritor = csv.writer(f)
            escritor.writerow([
                datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                correo["id"],
                correo["remitente"],
                correo["asunto"],
                correo["cuerpo"],
                "Requiere atención humana"
            ])
    else:
        print("   DECISIÓN: Respuesta automática generada:")
        print()
        # Indentamos la respuesta para que sea legible
        for linea in decision_ia.split("\n"):
            print(f"   {linea}")
        with open(ARCHIVO_RESPONDIDOS, mode="a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([
                datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                correo["id"],
                correo["remitente"],
                correo["asunto"],
                correo["cuerpo"],
                decision_ia
            ])
    print("-" * 60)

print("\n Triaje completado.")

---
## Probar el agente con un correo personalizado

Puedes escribir tu propio correo aquí y ver cómo responde el agente en tiempo real.

In [ ]:
# ✏️ Escribe aquí tu correo de prueba
mi_asunto = "Quiero saber si aceptáis Bizum"
mi_mensaje = "Hola, me gustaría hacer un pedido pero solo tengo Bizum. ¿Lo aceptáis como método de pago?"

print(f" Correo de prueba:")
print(f"  Asunto:  {mi_asunto}")
print(f"  Mensaje: {mi_mensaje}")
print(f"\n  Procesando...\n")

# La cadena LCEL devuelve str directamente
decision = rag_chain.invoke({
    "asunto":  mi_asunto,
    "mensaje": mi_mensaje
}).strip()

if "ESCALAR_A_HUMANO" in decision:
    print("  DECISIÓN: ESCALAR_A_HUMANO")
    print("   (El agente no encontró respuesta en el contexto o detectó urgencia)")
else:
    print(" DECISIÓN: Respuesta automática")
    print()
    print(decision)

---
## 📊 Resumen de la Fase 3

| Paso | Acción | Resultado |
|------|--------|-----------|
| 1 | Instalación de librerías adicionales| Conector con IA local |
| 2 | Verificación Ollama | Servidor local activo |
| 3 | Bandeja de entrada simulada | correos.json |
| 4 | Conectar con el RAG | Cargar la base de datos ChromaDB |
| 5 | Carga LLaMA3 vía Ollama | Modelo de lenguaje local activo |
| 6 | System Prompt | Reglas del agente definidas |
| 7 | Cadena RAG | Retriever + LLM encadenados |
| 8 | Bucle de triaje | Correos procesados y clasificados |
| * | Probar el agente con un correo personalizado | 

---

### 🏗️ Arquitectura completa del sistema
```
┌─────────────────────────────────────────────────┐
│              AGENTE DE TRIAJE                   │
│                                                 │
│  correos.json  ──►  RAG Chain  ──►  Decisión    │
│                        │                        │
│              ┌─────────┴─────────┐              │
│         ChromaDB            Ollama/LLaMA3       │
│         (faqs.txt)          (razonamiento)      │
│              └─────────┬─────────┘              │
│                        │                        │
│          ┌─────────────┴────────────┐           │
│    ✅ Respuesta              ⚠️  CSV escalados  │
│       automática              (humano revisa)   │
└─────────────────────────────────────────────────┘
```